# 文件与路径

学习目标：能用路径对象组织文件操作，区分文本、编码和字节数据，并正确读写、定位、复制与释放文件资源。

前置知识：字符串与容器、对象可变性、循环、函数与参数、模块导入、方法调用和异常处理。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

后续代码沿用前面已经导入的名称。每个涉及文件的代码单元都会新建 TemporaryDirectory，在其中生成小型输入，结束时自动清理。

## 1 在临时目录里读写第一份文件

Path 表示文件系统路径。用 / 可以连接目录与文件名；路径对象本身不是已打开的文件。Path.open 返回文件对象，"w" 表示写入，"r" 表示读取，encoding="utf-8" 指定文本编码。

这里先使用 with 管理资源：as 后的名称接收要使用的对象，缩进块结束后关闭文件。TemporaryDirectory 创建临时目录，其 with 块结束后移除目录及内容；先退出内层文件块，再清理外层目录。

本章先学习这种基本用法；第 7 节补充异常和关闭的关系，第 12 节介绍临时文件，完整的上下文管理协议留到第 15 章“上下文管理器”。

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    root = Path(directory)
    note = root / "note.txt"

    with note.open("w", encoding="utf-8", newline="\n") as stream:
        count = stream.write("你好\n")
    print(count, stream.closed)  # 3 True：写入的字符数，文件已关闭。

    with note.open("r", encoding="utf-8") as stream:
        print(repr(stream.read()))  # '你好\n'；repr 让换行可见。

print(root.exists())  # False：临时目录和其中的文件已经清理。

3 True
'你好\n'
False


## 2 路径、相对位置与目录

### 2.1 构造路径不等于创建文件

Path 按当前操作系统的规则处理路径。固定相对路径写在一个完整字符串中；有基目录变量时，用 / 或 joinpath 一次连接完整的相对后缀。右侧是绝对路径时，前面的路径会被舍弃。

下面 path 表示一个路径对象。这些属性和方法只处理路径的组成，with_name 与 with_suffix 不会重命名磁盘上的文件；真正的重命名放在第 13 节。

| 原文名称 | 中文名称／含义 |
| --- | --- |
| path.parts | 路径各部分组成的元组 |
| path.parent | 逻辑上的父路径 |
| path.name | 最后的路径部分 |
| path.stem | 去掉最后一个扩展名的名称 |
| path.suffix | 最后一个扩展名，含点号 |
| path.suffixes | 按顺序列出各扩展名 |
| path.joinpath | 连接路径部分 |
| path.with_name | 返回替换末尾名称后的新路径 |
| path.with_suffix | 返回替换最后一个扩展名后的新路径 |
| path.as_posix | 用正斜杠表示路径的字符串 |

In [2]:
path = Path("notes/archive.tar.gz")
print(path.parts)  # ('notes', 'archive.tar.gz')
print(path.parent.name, path.name)  # notes archive.tar.gz
print(path.stem, path.suffix)  # archive.tar .gz
print(path.suffixes)  # ['.tar', '.gz']
print(path == path.parent.joinpath(path.name))  # True
print(path.with_name("report.txt").as_posix())  # notes/report.txt
print(path.with_suffix(".xz").as_posix())  # notes/archive.tar.xz
print(path.name)  # archive.tar.gz：原路径对象没有被改变。

('notes', 'archive.tar.gz')
notes archive.tar.gz
archive.tar .gz
['.tar', '.gz']
True
notes/report.txt
notes/archive.tar.xz
archive.tar.gz


### 2.2 相对路径以工作目录为起点

相对路径交给 open 等文件操作时，相对于进程当前工作目录解释。Path.cwd 取得这个目录；Path.home 取得用户主目录，expanduser 才会把路径开头的 ~ 展开成主目录。

is_absolute 判断是否是绝对路径。Windows 的 C:notes 是盘符相关的相对路径，C:/notes 才是包含盘符和根目录的绝对路径；PureWindowsPath 可以只按 Windows 规则分析字符串，不访问磁盘。

resolve 返回绝对路径，解析符号链接并消除 ..。默认 strict=False 不要求最终路径存在；strict=True 遇到不存在的部分会报错。符号链接是指向其他路径的文件系统链接。

relative_to 默认只计算下级路径相对于给定上级路径的表示，不读取文件；不满足这种关系时引发 ValueError。它和 is_relative_to 都属于路径组成判断，不能把未解析的 .. 当作已经通过实际目录范围检查。

In [3]:
from pathlib import PureWindowsPath

relative = Path("notes/lesson.txt")
current = Path.cwd()
print(relative.is_absolute(), (current / relative).is_absolute())
# False True：连接到当前工作目录后得到绝对路径。
print(Path("~").expanduser() == Path.home())  # True
print(PureWindowsPath("C:notes").is_absolute())  # False
print(PureWindowsPath("C:/notes").is_absolute())  # True

# 下面只在临时目录内比较“规范化路径”和“文件存在”两个问题。
with TemporaryDirectory() as directory:
    root = Path(directory).resolve()
    (root / "scratch").mkdir()
    candidate = root / "scratch/../lesson.txt"
    resolved = candidate.resolve()
    print(resolved == root / "lesson.txt", resolved.exists())  # True False
    print(resolved.relative_to(root).as_posix())  # lesson.txt
    print(resolved.is_relative_to(root))  # True
    print(root / resolved == resolved)  # True：右侧绝对路径替换左侧。
    try:
        candidate.resolve(strict=True)
    except FileNotFoundError:
        print("文件尚未创建")  # resolve 默认成功不代表文件存在。

False True
True
False
True
True False
lesson.txt
True
True
文件尚未创建


### 2.3 创建、查询与筛选目录内容

mkdir 的 parents=True 会补建缺失的父目录；exist_ok=True 允许目标目录已经存在，但不允许同名普通文件占据该位置。touch 可创建空文件；对已有文件调用时，默认更新时间而不清空内容。

iterdir 只遍历直接子项，顺序没有保证。glob 用模式筛选路径，rglob 递归筛选；\*.txt 表示名称以 .txt 结尾，\*\* 可匹配当前层及下面的目录层级。匹配结果也可能是目录，需要普通文件时再调用 is_file。

| 原文名称 | 中文名称／含义 |
| --- | --- |
| Path.mkdir | 创建目录 |
| Path.touch | 创建文件或更新已有文件的时间 |
| Path.exists | 检查路径是否存在 |
| Path.is_file | 判断是否为普通文件 |
| Path.is_dir | 判断是否为目录 |
| Path.iterdir | 遍历直接子项 |
| Path.glob | 按相对模式匹配 |
| Path.rglob | 递归匹配 |
| Path.stat | 获取文件状态；普通文件的 st_size 是字节数 |

exists、is_file 和 is_dir 默认会跟随符号链接；查询结果反映查询时的状态，不能代替随后打开文件时的异常处理。

In [4]:
with TemporaryDirectory() as directory:
    root = Path(directory)
    reports = root / "notes/reports"
    reports.mkdir(parents=True, exist_ok=True)
    notes = reports.parent
    (notes / "a.txt").touch()
    (reports / "b.txt").touch()
    (notes / "folder.txt").mkdir()

    print(sorted([item.name for item in notes.iterdir()]))
    # ['a.txt', 'folder.txt', 'reports']；排序后输出固定。
    # glob 先匹配名称，再用 is_file 排除同名后缀的目录。
    direct_files = []
    for item in notes.glob("*.txt"):
        if item.is_file():
            direct_files.append(item.name)
    print(sorted(direct_files))  # ['a.txt']；排除了同样匹配的目录。

    # rglob 递归进入子目录，输出相对于 notes 的路径以区分文件。
    all_files = []
    for item in notes.rglob("*.txt"):
        if item.is_file():
            all_files.append(item.relative_to(notes).as_posix())
    print(sorted(all_files))  # ['a.txt', 'reports/b.txt']
    print(reports.is_dir(), (notes / "a.txt").stat().st_size)  # True 0

['a.txt', 'folder.txt', 'reports']
['a.txt']
['a.txt', 'reports/b.txt']
True 0


## 3 文件模式决定创建、覆盖和读写权限

### 3.1 r、w、a、x 与文本、二进制选项

文件模式是传给 open 的字符串：选择一种基本操作，再组合文本或二进制选项，必要时加入 +。默认模式 r 等同于 rt。

| 原文名称 | 中文名称／含义 | 已有文件与缺失文件的处理 |
| --- | --- | --- |
| r | 读取 | 要求文件存在 |
| w | 写入 | 打开时立即清空已有内容；不存在则创建 |
| a | 追加写入 | 写到末尾；不存在则创建 |
| x | 排他创建并写入 | 仅在目标不存在时创建，已有目标引发 FileExistsError |
| b | 二进制模式 | 读写字节数据，不做编码和换行转换 |
| t | 文本模式 | 读写字符串，是默认选项 |
| + | 更新模式 | 同时允许读取和写入，保留基本操作的创建或截断规则 |

例如 rb 读取二进制，wt 写入文本。b 和 t 二选一；二进制模式不传 encoding、errors 或 newline。父目录仍须存在，open 不会自动补建目录。

w 的清空发生在打开时，即使还没有调用 write。需要保留已有文件时，用 x 让创建动作本身拒绝覆盖，比先 exists 再用 w 更符合这一要求。

In [5]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "record.txt"
    try:
        with path.open("r", encoding="utf-8") as stream:
            stream.read()
    except FileNotFoundError:
        print("r：缺失文件不能读取")

    # 准备已有内容，随后依次对比覆盖、追加和独占创建。
    path.write_text("旧内容", encoding="utf-8")
    with path.open("w", encoding="utf-8", newline="\n"):
        pass
    print(repr(path.read_text(encoding="utf-8")))  # ''：打开 w 即已清空。

    with path.open("a", encoding="utf-8", newline="\n") as stream:
        stream.write("甲")
    with path.open("a", encoding="utf-8", newline="\n") as stream:
        stream.write("乙")
    print(path.read_text(encoding="utf-8"))  # 甲乙：第二次追加保留第一次。

    # 另取文件名观察 x：首次创建成功，第二次创建应失败。
    created = path.with_name("created.txt")
    with created.open("x", encoding="utf-8") as stream:
        stream.write("保留")
    try:
        with created.open("x", encoding="utf-8"):
            pass
    except FileExistsError:
        print("x：拒绝覆盖")
    print(created.read_text(encoding="utf-8"))  # 保留

r：缺失文件不能读取
''
甲乙
x：拒绝覆盖
保留


### 3.2 + 允许双向读写，不代表自动追加或自动清空

r+ 要求文件存在，不会在打开时截断；w+ 会先截断或创建，再允许双向读写。a+ 仍按追加方式写入，x+ 仍要求新建文件。

读写操作会推进当前位置。需要切换到指定位置时，用 seek；seek(0) 回到开头。r+ 在当前位置写入会覆盖对应内容，而不是向中间插入字符，也不会自动删掉剩余尾部。

truncate 不传参数时把文件大小改为当前位置。下面只用每个字符占一个 UTF-8 字节的英文字母；多字节文本的定位限制见第 9 节。

In [6]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "update.txt"
    path.write_text("abcdef", encoding="utf-8")

    # 先覆盖前几个字符，再显式截断；r+ 不会自动移除旧尾部。
    with path.open("r+", encoding="utf-8") as stream:
        stream.write("XY")
        stream.seek(0)
        print(stream.read())  # XYcdef：短写入仍保留原来的尾部。

        stream.seek(0)
        stream.write("Z")
        stream.truncate()
        stream.seek(0)
        print(stream.read())  # Z：显式截断移除了剩余尾部。

    with path.open("w+", encoding="utf-8") as stream:
        print(repr(stream.read()))  # ''：w+ 打开时清空已有内容。
        stream.write("new")
        stream.seek(0)
        print(stream.read())  # new：写入后先回到开头，再读取。

XYcdef
Z
''
new


## 4 Unicode、编码与字节数

### 4.1 str 保存码点序列，编码把文本转换成字节

Unicode 用码点（code point）标识字符；str 保存码点序列，len(str) 统计码点数，不是屏幕上的字形数。多个码点可以组合为一个字形，一个码点也不必单独显示为完整字形。

编码规定文本如何表示为字节：str.encode 得到 bytes，bytes.decode 按指定编码还原文本。为什么同一段文字会有两个不同的长度？分别数图中上下两层。

![同一段文字：3 个码点，8 个 UTF-8 字节](image/illustration/12-01-unicode-utf8.svg)

图示：A中😀 的 UTF-8 编码示意。方框宽度只帮助区分分组，不代表 Python 内部存储布局。

UTF-8 对 ASCII 码点使用一个字节，其他可编码码点使用二到四个字节；不能笼统地说所有汉字都占相同字节数。下面用 len 与 hex 核对 A中😀，再比较组合重音与另一种编码，观察码点、字形和字节数的区别。

In [7]:
text = "A中😀"
encoded = text.encode("utf-8")
print(len(text), len(encoded))  # 3 8：本例分别使用 1、3、4 个 UTF-8 字节。
print(encoded.hex(" "))  # 41 e4 b8 ad f0 9f 98 80
print(encoded.decode("utf-8") == text)  # True：按同一编码还原。

composed = "\u00e9"
combined = "e\u0301"
print(len(composed), len(combined))  # 1 2：可显示为相同的带重音字形。
print(composed == combined)  # False：码点序列不同。
print(len("中".encode("utf-8")), len("中".encode("utf-16-le")))
# 3 2：同一文本在不同编码下的字节数可以不同。

3 8
41 e4 b8 ad f0 9f 98 80
True
1 2
False


3 2


### 4.2 编码错误与解码错误是两个方向的问题

向不支持某些文本的编码转换，会引发 UnicodeEncodeError；字节序列不符合选定编码时，解码会引发 UnicodeDecodeError。encode、decode 以及文本 open 的 errors 默认采用 strict，发现错误就报告。

解码时 replace 用替换字符代替错误部分，ignore 则直接丢掉错误部分；它们都可能损失原始信息。编码时 replace 常用问号替换不能编码的字符，不应把两种方向的替换规则混为一谈。

选择错误编码不一定报错：同一串字节可能在另一种编码中也合法，却被解释成不同的文字。应根据数据来源约定指定 encoding，而不是把“不报错”当成编码正确。

In [8]:
# 预期 UnicodeEncodeError：直接观察原始异常，之后继续运行下一单元。
"中".encode("ascii")

UnicodeEncodeError: 'ascii' codec can't encode character '\u4e2d' in position 0: ordinal not in range(128)

In [9]:
damaged = b"A\xffB"

In [10]:
# 预期 UnicodeDecodeError：直接观察原始异常，之后继续运行下一单元。
damaged.decode("utf-8")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xff in position 1: invalid start byte

In [11]:
print(damaged.decode("utf-8", errors="replace"))  # A�B：错误位置被替换。
print(damaged.decode("utf-8", errors="ignore"))  # AB：原来的一个字节丢失。
print("中".encode("ascii", errors="replace"))  # b'?'

encoded = "é".encode("utf-8")
print(encoded.decode("latin-1") == "é")  # False：没有报错，却解码成了别的文本。

A�B
AB
b'?'
False


## 5 bytes、bytearray 与 memoryview

### 5.1 不可变字节序列与可变字节序列

bytes 与 bytearray 都是序列；这里的元素是 0 到 255（含端点）的整数，不是长度为 1 的字符串。bytes 不可变，bytearray 可变；转换为 bytes 可得到当前字节内容的不可变表示。

b 前缀表示字节字面量，直接书写的部分只允许 ASCII 字符，其他值用转义表示。例如 b"\x00" 含一个值为 0 的字节，b"" 是空字节串；汉字应先写成字符串，再调用 encode。

| 原文名称 | 中文名称／含义 |
| --- | --- |
| bytes | 不可变字节序列 |
| bytearray | 可变字节序列 |
| memoryview | 内存视图，通过缓冲区访问已有数据 |

bytes 的整数索引得到一个整数，切片仍得到 bytes。bytes(3) 创建三个零字节，不是把整数 3 转成文字；要表达数字文本，可先用 str 转换再编码。

In [12]:
payload = b"A\x00\xff"
print(payload[0], payload[:1], len(payload))  # 65 b'A' 3
print(bytes(3))  # b'\x00\x00\x00'
print(bytes([0, 255]).hex(" "))  # 00 ff

65 b'A' 3
b'\x00\x00\x00'
00 ff


In [13]:
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
payload[0] = 66

TypeError: 'bytes' object does not support item assignment

In [14]:
# 预期 ValueError：直接观察原始异常，之后继续运行下一单元。
bytes([256])

ValueError: bytes must be in range(0, 256)

In [15]:
editable = bytearray(b"cat")
editable[0] = ord("b")
snapshot = bytes(editable)
editable.append(ord("!"))
print(snapshot, editable)  # b'bat' bytearray(b'bat!')
# snapshot 保留转换时的内容，后续修改发生在 editable 中。

b'bat' bytearray(b'bat!')


### 5.2 memoryview 共享数据，不是副本

memoryview 引用支持缓冲区协议的对象，让代码直接访问其底层数据；缓冲区协议是对象暴露内存数据的一种接口。本例的 bytes 和 bytearray 都支持它。

对 bytearray 建立的视图可以修改原数据；对 bytes 建立的视图是只读的。切片产生子视图，不复制对应字节。视图的元素大小取决于原对象，本例每个元素一个字节，其他缓冲区类型不一定如此。

视图有效期间，bytearray 不能改变大小。release 或退出视图的 with 块会释放该视图；所有相关视图都释放后，才能解除它们造成的扩容限制。

In [16]:
buffer = bytearray(b"abcd")
# window 引用同一缓冲区的一段；修改会直接反映在 buffer 上。
with memoryview(buffer) as view:
    with view[1:3] as window:
        window[0] = ord("Z")
        print(bytes(buffer), window.readonly)  # b'aZcd' False
        try:
            buffer.append(ord("!"))
        except BufferError:
            print("有效视图存在时不能扩容")

buffer.append(ord("!"))
print(bytes(buffer))  # b'aZcd!'：内外两个视图都已释放。

# 换成不可变 bytes，观察只读视图与前一段可写视图的区别。
with memoryview(b"ab") as readonly_view:
    print(readonly_view.readonly)  # True
    try:
        readonly_view[0] = ord("Z")
    except TypeError:
        print("只读视图不能写入")

b'aZcd' False
有效视图存在时不能扩容
b'aZcd!'
True
只读视图不能写入


In [17]:
# 预期 ValueError：直接观察原始异常，之后继续运行下一单元。
view[0]

ValueError: operation forbidden on released memoryview object

## 6 文本流、二进制流与换行

### 6.1 选择模式就是选择读写接口

文本流读取后返回 str，写入时接收 str；二进制流读取后返回 bytes，写入时接收 bytes、bytearray 等类字节对象，不自动编码、解码或转换换行。连续的字节 memoryview 也可作为二进制写入数据。

Path.read_text 与 write_text 适合一次性处理小文本，read_bytes 与 write_bytes 适合小型二进制数据。它们负责打开和关闭文件；两种 write 方法都会覆盖已有文件。

文件扩展名不决定 Python 的读写模式。文本的 write 返回写入的字符数，二进制 write 返回写入的字节数；磁盘上的普通文件大小按字节计算。

In [18]:
with TemporaryDirectory() as directory:
    root = Path(directory)
    text_path = root / "text.dat"
    count = text_path.write_text("中\n", encoding="utf-8", newline="\n")
    print(count, text_path.stat().st_size)  # 2 4：字符数与 UTF-8 文件字节数。
    print(type(text_path.read_text(encoding="utf-8")).__name__)  # str
    print(type(text_path.read_bytes()).__name__)  # bytes

    # 扩展名不决定流类型：wb 要求字节，下面的 w 要求字符串。
    binary_path = root / "binary.txt"
    with binary_path.open("wb") as stream:
        try:
            stream.write("中")
        except TypeError:
            print("二进制写入需要字节数据")
        print(stream.write(bytearray([0, 255])))  # 2：接受可变字节序列。

    with text_path.open("w", encoding="utf-8") as stream:
        try:
            stream.write(b"ABC")
        except TypeError:
            print("文本写入需要字符串")
        stream.write("ABC")

2 4
str
bytes
二进制写入需要字节数据
2
文本写入需要字符串


### 6.2 顺序读取、空行与文件末尾

下面 size 表示一次最多读取的数量，是正整数；文本 read(size) 按字符计数，二进制按字节计数。省略 size 或传负数会读取剩余全部内容，适合已知很小的输入。

readline 返回下一行，并保留读到的行结束符；文件末尾没有换行时，最后一行也不会凭空补换行。文本读取在文件末尾返回 ""，空行则通常是 "\n"，二者不能混淆。

for 可以逐行处理文件对象；readlines 会把剩余各行一次性放入列表。这里先使用逐行接口，迭代器的完整机制放在下一章。

In [19]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "lines.txt"
    path.write_text("甲\n\n乙", encoding="utf-8", newline="\n")

    with path.open("r", encoding="utf-8") as stream:
        print(repr(stream.read(1)))  # '甲'
        print(repr(stream.readline()))  # '\n'：第一行还没读完的换行。
        print(repr(stream.readline()))  # '\n'：真正的空行。
        print(repr(stream.readline()))  # '乙'：末行没有换行符。
        print(repr(stream.readline()))  # ''：到达文件末尾。

    with path.open("r", encoding="utf-8") as stream:
        for number, line in enumerate(stream, start=1):
            print(number, repr(line))
        # 依次为 1 '甲\n'、2 '\n'、3 '乙'。
        print(stream.readlines())  # []：当前位置已在文件末尾。

'甲'
'\n'
'\n'
'乙'
''
1 '甲\n'
2 '\n'
3 '乙'
[]


### 6.3 换行识别与换行转换

newline 控制文本流如何识别或转换行结束符。下表中的 \n 表示换行符，\r 表示回车符，\r\n 是两个控制字符组成的行结束序列。

| newline 的值 | 中文名称／含义 | 读取规则 | 写入规则 |
| --- | --- | --- | --- |
| None | 默认换行处理 | 将 \n、\r、\r\n 都识别为行尾并返回 \n | 把 \n 转成系统默认行结束符 |
| "" | 通用识别但保留原样 | 识别以上三种行尾，不转换 | 不转换 |
| "\n" | 指定换行符 | 只以 \n 结束一行，不转换 | 不转换 |
| "\r" | 指定回车符 | 只以 \r 结束一行，不转换 | 把 \n 转成 \r |
| "\r\n" | 指定回车与换行序列 | 只以 \r\n 结束一行，不转换 | 把 \n 转成 \r\n |

write 和 writelines 都不会自动添加行结束符；writelines 只是依次写入给定字符串。print 默认会补一个 \n，可以用 file 指定文本流。下面同时观察读取时的规范化与写入时的转换。

In [20]:
with TemporaryDirectory() as directory:
    root = Path(directory)
    mixed = root / "mixed.txt"
    mixed.write_bytes(b"one\r\ntwo\rthree\n")

    with mixed.open("r", encoding="utf-8", newline=None) as stream:
        print(repr(stream.read()))  # 'one\ntwo\nthree\n'
    with mixed.open("r", encoding="utf-8", newline="") as stream:
        print(repr(stream.read()))  # 'one\r\ntwo\rthree\n'
    with mixed.open("r", encoding="utf-8", newline="\n") as stream:
        print(stream.readlines())  # ['one\r\n', 'two\rthree\n']

    output = root / "output.txt"
    with output.open("w", encoding="utf-8", newline="\r\n") as stream:
        stream.writelines(["甲", "乙\n"])
        print("丙", file=stream)  # 写入文件，不显示在单元输出；丙后跟随 CRLF。
    print(repr(output.read_bytes().decode("utf-8")))
    # '甲乙\r\n丙\r\n'：甲后没有补换行，两个 \n 都被转为 \r\n。

'one\ntwo\nthree\n'


'one\r\ntwo\rthree\n'
['one\r\n', 'two\rthree\n']
'甲乙\r\n丙\r\n'


## 7 with、缓冲与关闭后的边界

文件对象的 with 块在正常结束或发生异常时都会关闭文件。文件对象本身不会把块内异常自动吞掉；异常仍可交给外层的 except 处理。

flush 把 Python 流中的写缓冲交给下层，文件仍保持打开；close 会刷新并关闭，closed 可以检查关闭状态。flush 或 close 不能等同于断电后仍持久保存的保证，涉及磁盘同步还有更低层的操作。

不要依赖对象何时被回收来保存内容，也不要在关闭后继续读写。with 负责资源释放，不提供事务回滚：异常发生前已经写入的内容不会因此自动恢复。

In [21]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "partial.txt"
    try:
        with path.open("w", encoding="utf-8", newline="\n") as stream:
            stream.write("已写入\n")
            stream.flush()
            print(stream.closed)  # False：刷新没有关闭文件。
            raise ValueError("演示写入后的中断")
    except ValueError as error:
        print(error)  # 演示写入后的中断；异常传播到了外层。

    print(stream.closed)  # True：异常路径也关闭了文件。
    print(repr(path.read_text(encoding="utf-8")))  # '已写入\n'：不会回滚。
    try:
        stream.read()
    except ValueError:
        print("关闭后的文件不能继续读取")
    stream.close()  # 再次 close 不产生额外作用。

False
演示写入后的中断
True
'已写入\n'
关闭后的文件不能继续读取


## 8 分块读写

### 8.1 用有限大小的字节块复制文件

一次性 read 会把剩余数据全部装入内存。较大的文件可以反复 read(block_size)，立即处理或写出本次数据；block_size 是每块最多读取的字节数，本例取正整数 4。

对本章使用的普通、阻塞式二进制文件，read 返回 b"" 表示文件末尾，末块可以不足指定大小。不要用 read(0) 驱动复制循环，因为它本来就返回空字节串。

内层用同一个 with 管理输入、输出文件，块结束时两者都会关闭。这里使用默认缓冲的普通文件写入接口，不把这段循环推广成非阻塞网络读写规则。

In [22]:
with TemporaryDirectory() as directory:
    root = Path(directory)
    source = root / "source.bin"
    target = root / "target.bin"
    source.write_bytes(b"\x00AB\n\xffCDEF")

    total = 0
    with source.open("rb") as reader, target.open("xb") as writer:
        while True:
            chunk = reader.read(4)
            if chunk == b"":
                break
            total += writer.write(chunk)
            print(len(chunk))  # 依次 4、4、1，最后一块不足 4 字节。
    print(total)  # 9：累计写入的字节数。
    print(source.read_bytes() == target.read_bytes())  # True
    # 最后的整文件比较只用于本例的小输入，不是复制循环的必需步骤。

4
4
1
9
True


### 8.2 文本流负责跨字节块的解码

多字节编码的边界不一定与 read 的字节块边界重合。任意截取一块 UTF-8 字节再单独 decode，可能刚好截断一个码点的编码。

处理文本时，交给指定 encoding 的文本流读取；它负责跨底层字节块解码。文本 read(size) 限制返回的字符数，不表示固定的磁盘字节数。只搬运数据、无需理解文本时，则保留二进制复制方式。

In [23]:
with TemporaryDirectory() as directory:
    root = Path(directory)
    source = root / "source.txt"
    target = root / "lower.txt"
    source.write_text("中文AB", encoding="utf-8")

    # 先故意按字节截断中文，再用文本流按字符读取作对照。
    with source.open("rb") as reader:
        incomplete = reader.read(2)
    try:
        incomplete.decode("utf-8")
    except UnicodeDecodeError:
        print("两个字节截断了“中”的 UTF-8 编码")

    with (
        source.open("r", encoding="utf-8") as reader,
        target.open("x", encoding="utf-8", newline="\n") as writer,
    ):
        # reader 负责跨底层字节块解码；空字符串表示文本已读完。
        while True:
            chunk = reader.read(2)
            if chunk == "":
                break
            print(repr(chunk))  # 依次 '中文'、'AB'，没有半个码点。
            writer.write(chunk.lower())
    print(target.read_text(encoding="utf-8"))  # 中文ab

两个字节截断了“中”的 UTF-8 编码
'中文'
'AB'
中文ab


## 9 用 seek 与 tell 定位

### 9.1 二进制文件按字节定位

tell 返回二进制文件当前位置到文件开头的字节距离。seek(offset, whence) 调整位置，offset 表示相对参考点的字节偏移，whence 选择参考点；seek 返回调整后的绝对位置。

| whence 的值 | 中文名称／含义 | offset 的常见取值 |
| --- | --- | --- |
| 0 | 文件开头，也是默认值 | 非负字节偏移 |
| 1 | 当前位置 | 可前移或后移，但最终位置不能为负 |
| 2 | 文件末尾 | 常用零或负偏移 |

是否支持定位可用 seekable 查询；并非所有流都能定位。下面使用普通文件的 r+b，观察读写都发生在当前位置，写入覆盖对应字节而不插入内容。

In [24]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "positions.bin"
    path.write_bytes(b"abcdef")
    with path.open("r+b") as stream:
        print(stream.seekable())  # True
        print(stream.read(2), stream.tell())  # b'ab' 2
        stream.seek(1, 1)
        print(stream.read(1))  # b'd'：从位置 2 再前进一个字节。
        stream.seek(-1, 2)
        print(stream.read(1))  # b'f'：定位到末尾前一个字节。
        stream.seek(2)
        stream.write(b"XY")
    print(path.read_bytes())  # b'abXYef'：覆盖两个字节，总长度不变。

True
b'ab' 2
b'd'
b'f'
b'abXYef'


### 9.2 文本位置是用于恢复状态的值

文本 tell 返回不透明的位置值，也称 cookie：应保存并交回同一文本流的 seek 来恢复读取位置，不把它解释为字符下标或普通字节数，更不能随意加减。

对文本文件，使用下列受支持的形式。这里 position 表示从该文本流 tell 取得的位置值，不是自行计算的整数。

| 写法 | 中文名称／含义 |
| --- | --- |
| seek(0) | 回到开头 |
| seek(position) | 恢复先前保存的位置 |
| seek(0, 1) | 保持当前位置 |
| seek(0, 2) | 到文件末尾 |

非零的当前位置或末尾相对偏移不受支持；任意拼出的开头偏移也不符合文本接口的约定。文本定位不仅要考虑编码，还可能要恢复换行解码状态。

In [25]:
import io

with TemporaryDirectory() as directory:
    path = Path(directory) / "positions.txt"
    path.write_bytes("甲\r\n乙\n".encode("utf-8"))
    with path.open("r", encoding="utf-8") as stream:
        print(repr(stream.readline()))  # '甲\n'：CRLF 已被规范化。
        # 保存文本流自己返回的位置标记，再用同一个标记恢复读取。
        position = stream.tell()
        second_line = stream.readline()
        stream.seek(position)
        print(stream.readline() == second_line)  # True：恢复到第二行开头。
        print(stream.seek(0, 1) == stream.tell())  # True：位置保持不变。
        stream.seek(0, 2)
        print(repr(stream.read()))  # ''：已经位于末尾。
        try:
            stream.seek(1, 1)
        except io.UnsupportedOperation:
            print("文本流不支持非零的当前位置相对偏移")
        stream.seek(0)
        print(stream.read(1))  # 甲

'甲\n'


True
True
''
文本流不支持非零的当前位置相对偏移
甲


## 10 用 StringIO 与 BytesIO 在内存中读写

StringIO 是内存文本流，BytesIO 是内存二进制流，适合组装小内容或给接收文件对象的函数提供小型输入；它们提供 read、write、seek 等接口，不需要磁盘路径。

StringIO 有初始内容时，位置在开头，直接写入会覆盖相应内容。getvalue 返回全部内容而不改变当前位置；流关闭后，其内部缓冲会被丢弃，需要结果时在关闭前取得它。

BytesIO.getbuffer 返回共享内存视图；有效视图存在时，BytesIO 不能改变大小或关闭。嵌套 with 可以先释放视图，再关闭流。这与前面 bytearray 的视图共享规则属于同一类缓冲区访问。

In [26]:
with io.StringIO("AB") as text_stream:
    text_stream.write("X")
    position = text_stream.tell()
    saved_text = text_stream.getvalue()
    print(saved_text, text_stream.tell() == position)  # XB True
    text_stream.seek(0)
    print(text_stream.read())  # XB：写入发生在初始位置。
print(saved_text, text_stream.closed)  # XB True：取出的字符串仍可使用。

with io.BytesIO(b"\x01\x02") as binary_stream:
    binary_stream.write(b"\xff")
    with binary_stream.getbuffer() as view:
        view[1] = 3
        print(binary_stream.getvalue().hex(" "))  # ff 03：视图修改了原缓冲。
    binary_stream.seek(0)
    saved_bytes = binary_stream.read()
print(saved_bytes.hex(" "), binary_stream.closed)  # ff 03 True

XB True
XB
XB True
ff 03
ff 03 True


## 11 标准输入、标准输出与标准错误

sys 提供解释器的标准流，它们用于连接输入来源和输出目的地。本章关注文件式接口，命令行参数等 sys 用法属于其他章节。

| 原文名称 | 中文名称／含义 |
| --- | --- |
| sys.stdin | 标准输入，用于读取输入；input 也使用它 |
| sys.stdout | 标准输出，print 默认写入这里 |
| sys.stderr | 标准错误，解释器错误消息及程序诊断通常写入这里 |

这些接口通常是文本流，读取可能等待输入。普通脚本可把 sys.stdin 传给下面函数；示例改传有确定内容的 StringIO，避免等待交互。输出和诊断分别写到实际的 sys.stdout 与 sys.stderr，不关闭调用者提供的流。

解释器通常让交互式 stdout 按行缓冲，非交互式 stdout 按块缓冲，stderr 按行缓冲；print 的 flush=True 可以请求刷新。宿主可替换标准流，不能假定它们总有 buffer 属性；只有确实提供该属性时，才可借助底层二进制流读写字节。

In [27]:
import sys


def report_line(input_stream, output_stream, error_stream):
    """读取一行，将正文或输入结束提示写到相应的文本流。"""
    line = input_stream.readline()
    if line == "":
        print("输入已结束", file=error_stream, flush=True)
        return
    print("收到：" + line.rstrip("\n"), file=output_stream, flush=True)


# 普通脚本可将 sys.stdin 作为第一个参数；此处用内存输入避免阻塞。
with io.StringIO("甲\n") as source:
    report_line(source, sys.stdout, sys.stderr)  # stdout：收到：甲
with io.StringIO("") as source:
    report_line(source, sys.stdout, sys.stderr)  # stderr：输入已结束
# stderr 输出是一条有意写出的提示，不是未捕获的异常。

收到：甲


输入已结束


## 12 tempfile 与自动清理

TemporaryDirectory 适合一组临时文件；默认在退出 with 时移除整个临时目录。文件应先关闭，再退出目录的 with，避免 Windows 上仍占用文件而无法清理。

TemporaryFile 返回可读写的临时文件，默认模式为 w+b，关闭后删除；不要依赖它在文件系统中是否有可见名称。NamedTemporaryFile 则保证有可见名称，可通过 name 获取路径。

NamedTemporaryFile 默认关闭即删除。Python 3.12 的 delete_on_close=False 在默认 delete=True 下，把删除延后到 with 退出；因此可以先关闭原文件，再按名称重新打开。重新打开的文件也要在外层退出前关闭。

In [28]:
import tempfile

with TemporaryDirectory() as directory:
    with tempfile.TemporaryFile(dir=directory) as stream:
        stream.write(b"temporary")
        stream.seek(0)
        print(stream.read())  # b'temporary'；默认模式允许双向二进制读写。

    # 显式关闭文件句柄后重开；外层 with 结束时才删除带名字的文件。
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        newline="\n",
        dir=directory,
        delete_on_close=False,
    ) as named:
        named_path = Path(named.name)
        named.write("暂存\n")
        named.close()
        print(named_path.exists())  # True：关闭时暂不删除。
        with named_path.open("r", encoding="utf-8") as reopened:
            print(repr(reopened.read()))  # '暂存\n'
    print(named_path.exists())  # False：退出 NamedTemporaryFile 的 with。

b'temporary'
True
'暂存\n'
False


## 13 重命名、复制、移动与删除

### 13.1 Path 的重命名与删除

rename 实际重命名文件或目录，返回指向目标的新 Path；旧 Path 对象的文字内容不会跟着变化。目标若已存在，Unix 与 Windows 的 rename 覆盖行为不同；示例只重命名到不存在的目标。

replace 明确替换已有目标文件。rename 和 replace 的相对目标路径相对于当前工作目录，不是相对于源文件的父目录；下面始终从临时根目录构造目标。

unlink 删除文件或符号链接，missing_ok=True 只忽略文件不存在；rmdir 删除空目录，非空目录不能用它删除。删除路径不会把旧 Path 对象变成别的路径。

| 原文名称 | 中文名称／含义 |
| --- | --- |
| Path.rename | 实际重命名，已有目标的处理取决于平台 |
| Path.replace | 重命名并替换已有目标文件 |
| Path.unlink | 删除文件或符号链接 |
| Path.rmdir | 删除空目录 |

In [29]:
with TemporaryDirectory() as directory:
    root = Path(directory).resolve()
    original = root / "original.txt"
    original.write_text("旧", encoding="utf-8")
    renamed = original.rename(root / "renamed.txt")
    print(original.name, original.exists(), renamed.exists())
    # original.txt False True：旧路径对象仍表示旧位置。

    replacement = root / "replacement.txt"
    replacement.write_text("新", encoding="utf-8")
    replacement.replace(renamed)
    print(renamed.read_text(encoding="utf-8"))  # 新：替换了目标原有内容。
    print(replacement.exists())  # False：替换过程移动了源文件。

    renamed.unlink()
    renamed.unlink(missing_ok=True)
    empty = root / "empty"
    empty.mkdir()
    empty.rmdir()
    print(renamed.exists(), empty.exists())  # False False

original.txt False True
新
False
False False


### 13.2 shutil 处理文件复制与目录树

shutil 提供较高层的文件操作。元数据（metadata）是权限、时间等描述文件的信息，与文件正文不同；copy2 只是尽量保留元数据，不能保证在所有平台完整保留。

| 原文名称 | 中文名称／含义 |
| --- | --- |
| shutil.copyfile | 复制文件内容，目标必须是完整文件名 |
| shutil.copy | 复制内容和权限模式，目标可为文件或目录 |
| shutil.copy2 | 在 copy 基础上尽量保留时间等元数据 |
| shutil.copytree | 递归复制目录树，默认要求目标目录不存在 |
| shutil.move | 移动文件或目录；跨文件系统时通过复制后删除实现 |
| shutil.rmtree | 删除整个目录树及其内容 |

前三种复制都可能覆盖已有目标文件。copytree 的 dirs_exist_ok=True 允许合并到已有目录，也会覆盖对应文件；不要把它理解成“只复制缺失文件”。复制不会自动删除源文件。

move 的目标若是已有目录，会把源移入该目录，且其中不能已有同名目标；其他已有目标的覆盖行为受底层重命名规则影响。下面所有操作都限制在新建的临时目录中，删除前再次检查具体目标属于该目录。

In [30]:
import shutil

with TemporaryDirectory() as directory:
    root = Path(directory).resolve()
    source_dir = root / "source"
    source_dir.mkdir()
    source = source_dir / "note.txt"
    source.write_text("资料", encoding="utf-8")

    # 分别观察只复制内容、同时复制模式、尽量复制元数据的接口。
    content_copy = root / "content.txt"
    content_copy.write_text("将被覆盖", encoding="utf-8")
    shutil.copyfile(source, content_copy)
    mode_copy = Path(shutil.copy(source, root / "mode.txt"))
    metadata_copy = Path(shutil.copy2(source, root / "metadata.txt"))
    print(content_copy.read_text(encoding="utf-8"))  # 资料：目标被覆盖。
    print(mode_copy.read_bytes() == metadata_copy.read_bytes())  # True
    # 这里只检查正文，不能据此宣称所有元数据都完整保留。

    # 目录树复制与单文件移动各用独立目标，便于观察原路径的变化。
    backup = root / "backup"
    shutil.copytree(source_dir, backup)
    print((backup / "note.txt").read_text(encoding="utf-8"))  # 资料
    moved = Path(shutil.move(content_copy, root / "moved.txt"))
    print(content_copy.exists(), moved.exists())  # False True
    print(source.exists())  # True：被移动的是副本，原文件仍在。

    # 递归删除前限定真实目标；这里的路径边界检查不能省略。
    checked_backup = backup.resolve()
    if (
        checked_backup == root
        or not checked_backup.is_relative_to(root)
        or not checked_backup.is_dir()
    ):
        raise ValueError("删除目标必须是临时根目录下的子目录")
    shutil.rmtree(checked_backup)
    print(backup.exists())  # False；其他临时内容由外层 with 清理。

资料
True
资料
False True
True
False


## 本章小结

（1）Path 表示位置，文件对象管理读写状态。相对路径以工作目录为起点，构造路径、解析路径和创建文件是不同操作。

（2）文件模式决定读取、创建、截断或追加；+ 只增加双向读写能力。with 负责关闭资源，不负责撤销已经发生的写入。

（3）str 的码点数、编码后的字节数与可见字形数不能混用。文本流处理编码和换行，二进制流保留字节；memoryview 共享已有缓冲区。

（4）逐行或分块处理可以避免一次装入整个文件。二进制按字节定位，文本只使用接口允许的位置恢复方式。

（5）StringIO、BytesIO 和标准流也提供文件式接口；临时资源与复制、移动、删除操作都要明确其生命周期和覆盖语义。

自查：能否说明为什么文本短写入可能留下尾部、错误编码可能不报错，以及文件关闭和写入回滚不是一回事？

## 练习

### 练习 1：预测覆盖与当前位置

先预测两次 print 的输出，再运行核对。解释写入是否改变总长度，以及第一次 read 完成后文件位置在哪里。不要通过运行前面的示例替代对这段代码的逐步推演。

In [31]:
# 先预测两次读取的字节内容，再核对覆盖写入与文件位置。
with TemporaryDirectory() as directory:
    path = Path(directory) / "prediction.bin"
    path.write_bytes(b"abcdef")
    with path.open("r+b") as stream:
        stream.write(b"XY")
        stream.seek(0)
        print(stream.read())
        print(stream.read(2))

b'XYcdef'
b''


### 练习 2：统一文本行结束符并拒绝覆盖

在一个新的 TemporaryDirectory 中，先把 "甲\r\n\r乙\n" 编码成 UTF-8 字节写入 input.txt。读取时统一行结束符，跳过空行，把非空行依次写成“序号:正文”，保存为 output.txt。

输出使用 UTF-8、\n 行结束符和 x 模式。完成后检查读取结果等于 "1:甲\n2:乙\n"，原始输入字节不变；再次以 x 打开同一输出时，只捕获预期的 FileExistsError，并确认输出仍保持原样。

所有文件必须在本题的临时目录内创建和检查，退出后自动清理。

In [32]:
# 在新的 TemporaryDirectory 内生成输入、处理文本并加入题目要求的检查。
# 当前占位不执行文件操作，可以从空内核顺序运行。
pass

### 练习 3：编写按块复制函数

编写 copy_chunks(source, target, block_size=4)：source 和 target 是文件路径，block_size 约定为整数，表示每次最多读取的字节数。参数不大于零时，在打开任何文件前引发 ValueError。

函数用 rb 读取、xb 创建目标，按块复制并返回实际写入的总字节数，不使用一次性整文件读取。已有目标应让 FileExistsError 传播，函数负责关闭自己打开的文件。

在新的 TemporaryDirectory 中检查空文件、长度不是块大小倍数的文件，以及包含 0、255 字节的文件；分别确认目标内容相同、返回值等于原字节数。再检查非法块大小不会创建目标，已有目标也不会被覆盖。

In [33]:
# 在此定义 copy_chunks，并在新的 TemporaryDirectory 中检查各项边界。
# 当前占位不定义会被误认为已实现的函数，也不创建文件。
pass

## 参考与引用来源

| 网站 | 版本、具体位置与支持内容 |
| --- | --- |
| Python 官方文档（docs.python.org） | Python 3.12：[pathlib 的路径组成（表中同名属性与方法）](https://docs.python.org/3.12/library/pathlib.html#methods-and-properties)；[路径拼接](https://docs.python.org/3.12/library/pathlib.html#operators)；[Path.cwd](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.cwd)；[Path.home](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.home)；[Path.expanduser](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.expanduser)；[PureWindowsPath](https://docs.python.org/3.12/library/pathlib.html#pathlib.PureWindowsPath)；[Path.resolve](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.resolve)；[Path.relative_to](https://docs.python.org/3.12/library/pathlib.html#pathlib.PurePath.relative_to)；[Path.is_relative_to](https://docs.python.org/3.12/library/pathlib.html#pathlib.PurePath.is_relative_to)；[Path.mkdir](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.mkdir)；[Path.touch](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.touch)；[Path.exists](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.exists)；[Path.is_file](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.is_file)；[Path.is_dir](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.is_dir)；[Path.iterdir](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.iterdir)；[Path.glob](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.glob)；[Path.rglob](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.rglob)；[Path.stat](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.stat)；[文件大小 st_size](https://docs.python.org/3.12/library/os.html#os.stat_result.st_size)；[open：模式、编码、错误处理及换行](https://docs.python.org/3.12/library/functions.html#open)；[教程：读写、当前位置与关闭](https://docs.python.org/3.12/tutorial/inputoutput.html#reading-and-writing-files)；[Unicode：码点与字形](https://docs.python.org/3.12/howto/unicode.html#definitions)；[Unicode：编码与 UTF-8](https://docs.python.org/3.12/howto/unicode.html#encodings)；[Unicode：编码、解码与错误处理](https://docs.python.org/3.12/howto/unicode.html#the-string-type)；[Unicode：组合字符与比较](https://docs.python.org/3.12/howto/unicode.html#comparing-strings)；[Unicode：分块解码](https://docs.python.org/3.12/howto/unicode.html#reading-and-writing-unicode-data)；[bytes](https://docs.python.org/3.12/library/stdtypes.html#bytes)；[bytearray](https://docs.python.org/3.12/library/stdtypes.html#bytearray)；[memoryview](https://docs.python.org/3.12/library/stdtypes.html#memoryview)；[memoryview.release](https://docs.python.org/3.12/library/stdtypes.html#memoryview.release)；[BufferError](https://docs.python.org/3.12/library/exceptions.html#BufferError)；[文本流](https://docs.python.org/3.12/library/io.html#text-i-o)；[二进制流](https://docs.python.org/3.12/library/io.html#binary-i-o)；[Path.read_text](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.read_text)；[Path.write_text](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.write_text)；[Path.read_bytes](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.read_bytes)；[Path.write_bytes](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.write_bytes)；[IOBase：关闭与文件接口](https://docs.python.org/3.12/library/io.html#io.IOBase)；[writelines](https://docs.python.org/3.12/library/io.html#io.IOBase.writelines)；[truncate](https://docs.python.org/3.12/library/io.html#io.IOBase.truncate)；[flush](https://docs.python.org/3.12/library/io.html#io.IOBase.flush)；[磁盘同步 fsync](https://docs.python.org/3.12/library/os.html#os.fsync)；[缓冲二进制读取](https://docs.python.org/3.12/library/io.html#io.BufferedIOBase.read)；[缓冲二进制写入](https://docs.python.org/3.12/library/io.html#io.BufferedIOBase.write)；[二进制 seek](https://docs.python.org/3.12/library/io.html#io.IOBase.seek)；[文本 seek](https://docs.python.org/3.12/library/io.html#io.TextIOBase.seek)；[文本 tell](https://docs.python.org/3.12/library/io.html#io.TextIOBase.tell)；[TextIOWrapper：换行与位置恢复](https://docs.python.org/3.12/library/io.html#io.TextIOWrapper)；[UnsupportedOperation](https://docs.python.org/3.12/library/io.html#io.UnsupportedOperation)；[StringIO](https://docs.python.org/3.12/library/io.html#io.StringIO)；[BytesIO](https://docs.python.org/3.12/library/io.html#io.BytesIO)；[BytesIO.getbuffer](https://docs.python.org/3.12/library/io.html#io.BytesIO.getbuffer)；[sys 标准流、缓冲及宿主替换](https://docs.python.org/3.12/library/sys.html#sys.stdin)；[TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)；[TemporaryFile](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryFile)；[NamedTemporaryFile 与 3.12 删除选项](https://docs.python.org/3.12/library/tempfile.html#tempfile.NamedTemporaryFile)；[Path.rename](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.rename)；[Path.replace](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.replace)；[Path.unlink](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.unlink)；[Path.rmdir](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.rmdir)；[shutil.copyfile](https://docs.python.org/3.12/library/shutil.html#shutil.copyfile)；[shutil.copy](https://docs.python.org/3.12/library/shutil.html#shutil.copy)；[shutil.copy2](https://docs.python.org/3.12/library/shutil.html#shutil.copy2)；[shutil.copytree](https://docs.python.org/3.12/library/shutil.html#shutil.copytree)；[shutil.move](https://docs.python.org/3.12/library/shutil.html#shutil.move)；[shutil.rmtree](https://docs.python.org/3.12/library/shutil.html#shutil.rmtree)。 |